In [1]:
# Step 1: Install dependencies & set Hugging Face token
%pip install -q "datasets>=2.19.0" "huggingface_hub>=0.24"
import os
import getpass

# 直接设置Hugging Face token，跳过登录界面
hf_token = getpass.getpass("Paste your Hugging Face token: ")
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Paste your Hugging Face token:  ········


Hugging Face token set successfully!


In [2]:
# Step 2: Load FLARE-FNXL test set and preprocess
from datasets import load_dataset, Dataset

# FLARE-FNXL dataset (Financial Numeric Extreme Labeling)
print("加载 FLARE-FNXL 数据集...")
ds_raw = load_dataset("ChanceFocus/flare-fnxl", split="test")
print(f"成功加载! 样本数: {len(ds_raw)}, 列: {ds_raw.column_names}")

# 定义标签类型（从数据集中提取的主要类别）
# FNXL 任务是对每个token进行金融数值极端标注
LABELS = ["O", "B-DeferredCompensationArrangementWithIndividualCompensationExpense", 
          "B-BusinessCombinationContingentConsiderationLiability", "B-InterestExpense",
          "B-DefinedBenefitPlanExpectedAmortizationOfGainLossNextFiscalYear",
          "B-ShareBasedCompensationArrangementByShareBasedPaymentAwardEquityInstrumentsOtherThanOptionsVestedInPeriod",
          "B-DefinedContributionPlanEmployerMatchingContributionPercent", "B-CostOfGoodsAndServicesSold",
          "B-ImpairmentOfIntangibleAssetsFinitelived", "B-LongtermDebtWeightedAverageInterestRate"]

def _map_row(x):
    text = x.get("text") or x.get("sentence") or ""
    tokens = x.get("token", [])
    labels = x.get("label", [])
    answer = x.get("answer", "")
    
    # 如果没有预分的 tokens，从 text 分词
    if not tokens and text:
        tokens = text.split()
    
    # 确保 tokens 和 labels 长度匹配
    if len(tokens) != len(labels):
        min_len = min(len(tokens), len(labels))
        tokens = tokens[:min_len]
        labels = labels[:min_len]
    
    return {
        "tokens": tokens,
        "labels": labels,
        "text": " ".join(tokens) if tokens else text,
        "answer": answer,
        "label_types": LABELS
    }

ds = Dataset.from_list([_map_row(r) for r in ds_raw])
bad = [i for i, r in enumerate(ds) if not r["tokens"]]
print(f"空样本数: {len(bad)}")
assert len(bad) == 0, "发现空样本。"

print(f"\n第一个样本示例:")
print(f"  Tokens: {ds[0]['tokens'][:10]}...")
print(f"  Labels: {ds[0]['labels'][:10]}...")

加载 FLARE-FNXL 数据集...


README.md:   0%|          | 0.00/523 [00:00<?, ?B/s]

C:\Users\klofu\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\klofu\.cache\huggingface\hub\datasets--ChanceFocus--flare-fnxl. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling 

(…)-00000-of-00001-35c7f3863a79cbce.parquet:   0%|          | 0.00/315k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/318 [00:00<?, ? examples/s]

成功加载! 样本数: 318, 列: ['id', 'query', 'answer', 'text', 'label', 'token']
空样本数: 0

第一个样本示例:
  Tokens: ['Compensation', 'expense', 'recognized', 'for', 'all', 'of', 'the', 'Company', "'s", 'deferred']...
  Labels: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']...


In [3]:
# Step 3: Install dependencies, configure OpenAI, and record experiment metadata
%pip install -q "openai==1.40.2" "httpx==0.27.2" "httpcore==1.0.5" \
               "pandas>=2.2.2" "tqdm>=4.66.4" "requests>=2.31.0"

import os, getpass, json, time, platform, re
from importlib.metadata import version, PackageNotFoundError
import requests

# o3-mini配置
MODEL = "o3-mini"
BASE_URL = "https://api.openai.com/v1"

api_key = os.getenv("OPENAI_API_KEY") or os.getenv("API_KEY")
if not api_key:
    api_key = getpass.getpass("Paste your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

# 文件命名
dataset_name = "flare_fnxl"
run_tag = f"{dataset_name}_{MODEL.replace('-', '_')}"
save_dir = "/content"
pred_path = f"{save_dir}/{run_tag}_predictions.csv"
meta_path = f"{save_dir}/{run_tag}_metadata.json"

def ver(pkg: str) -> str:
    try:
        return version(pkg)
    except PackageNotFoundError:
        return "not-installed"

meta = {
    "dataset": "ChanceFocus/flare-fnxl",
    "split": "test",
    "label_types": LABELS,
    "model": MODEL,
    "openai_sdk": ver("openai"),
    "httpx": ver("httpx"),
    "httpcore": ver("httpcore"),
    "datasets_version": ver("datasets"),
    "pandas": ver("pandas"),
    "tqdm": ver("tqdm"),
    "time_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "python": platform.python_version(),
    "base_url": BASE_URL,
    "note": "o3-mini evaluation on FLARE-FNXL (Financial Numeric Extreme Labeling)"
}

os.makedirs(save_dir, exist_ok=True)
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Meta saved ->", meta_path)
print("MODEL:", MODEL, "| BASE_URL:", BASE_URL)
print("OPENAI_API_KEY is set:", bool(os.environ.get("OPENAI_API_KEY")))

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Paste your OpenAI API key:  ········


Meta saved -> /content/flare_fnxl_o3_mini_metadata.json
MODEL: o3-mini | BASE_URL: https://api.openai.com/v1
OPENAI_API_KEY is set: True


In [4]:
# Step 3.5: 测试10个样本（集成Step4+Step5功能，自动评估）
print("="*60)
print("Step 3.5: 测试10个样本 - FLARE-FNXL")
print("自动评估模式 | 集成推理+评估")
print("="*60)

import json
import re
import time
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
from collections import Counter

def _strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z0-9_-]*\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    return s.strip()

def _make_fnxl_prompt(tokens):
    """创建FNXL任务的提示词"""
    text = " ".join(tokens)
    return (
        "Task: Perform Financial Numeric Extreme Labeling (FNXL) on the following text.\n\n"
        f"Text: {text}\n\n"
        "Instructions:\n"
        "1. For each token, identify if it belongs to any financial numeric category\n"
        "2. Label tokens with appropriate B-* categories (B- indicates beginning of a category)\n"
        "3. Use 'O' for tokens without special labels\n"
        "4. Return the labels as a JSON array in the same order as tokens\n"
        "5. Return ONLY the JSON array, no additional text\n\n"
        "Example output format:\n"
        '["O", "O", "B-InterestExpense", "O", "B-DeferredCompensationArrangementWithIndividualCompensationExpense"]'
    )

def parse_fnxl_response(response_text: str, num_tokens: int):
    """解析模型返回的token标签"""
    response_text = response_text.strip()
    match = re.search(r'\[.*\]', response_text, re.DOTALL)
    if match:
        try:
            labels = json.loads(match.group())
            if isinstance(labels, list):
                if len(labels) > num_tokens:
                    labels = labels[:num_tokens]
                elif len(labels) < num_tokens:
                    labels = labels + ["O"] * (num_tokens - len(labels))
                return labels, True, None
        except json.JSONDecodeError:
            pass
    return None, False, "Failed to parse JSON array"

def ask_fnxl_sample(tokens):
    """调用API获取预测标签"""
    url = f"{BASE_URL}/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    user_text = _make_fnxl_prompt(tokens)
    
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "You are a financial NLP expert. Respond only with valid JSON arrays."},
            {"role": "user", "content": user_text}
        ]
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload, timeout=60)
        
        if response.status_code != 200:
            return None, False, f"API Error: {response.status_code}"
            
        result = response.json()
        
        if 'choices' not in result or len(result['choices']) == 0:
            return None, False, "No response"
            
        txt = result['choices'][0]['message']['content']
        txt = _strip_code_fences(txt)
        pred_labels, success, error = parse_fnxl_response(txt, len(tokens))
        return pred_labels, success, error
        
    except Exception as e:
        return None, False, str(e)

# 测试前10个样本
test_size = min(10, len(ds))
print(f"\n自动评估前 {test_size} 个样本...\n")

all_true = []
all_pred = []
sample_results = []

for i in range(test_size):
    sample = ds[i]
    tokens = sample["tokens"]
    true_labels = sample["labels"]
    
    print(f"样本 {i}: 正在处理 ({len(tokens)} tokens)...")
    
    pred_labels, success, error = ask_fnxl_sample(tokens)
    
    if success and pred_labels:
        all_true.extend(true_labels)
        all_pred.extend(pred_labels)
        sample_results.append({
            "index": i,
            "status": "success",
            "token_count": len(tokens),
            "true_preview": true_labels[:5] if len(true_labels) > 5 else true_labels,
            "pred_preview": pred_labels[:5] if len(pred_labels) > 5 else pred_labels
        })
        print(f"  ✅ 成功 | 预测前5: {pred_labels[:5]}")
    else:
        # 失败时用O填充
        all_true.extend(true_labels)
        all_pred.extend(["O"] * len(tokens))
        sample_results.append({
            "index": i,
            "status": f"failed: {error}",
            "token_count": len(tokens),
            "true_preview": true_labels[:5] if len(true_labels) > 5 else true_labels,
            "pred_preview": ["O"] * 5
        })
        print(f"  ❌ 失败: {error}")
    
    # 避免API限流
    time.sleep(0.5)

# 计算评估指标
print("\n" + "="*60)
print("EVALUATION RESULTS - o3-mini on FLARE-FNXL (10个测试样本)")
print("="*60)

# 获取所有唯一的标签
unique_labels = sorted(set(all_true + all_pred))
print(f"\n总Token数: {len(all_true)}")
print(f"唯一标签数: {len(unique_labels)}")

accuracy = accuracy_score(all_true, all_pred)
print(f"\nToken-level Accuracy: {accuracy:.4f}")

# 显示前10个最常见的标签
label_counts = Counter(all_true)
print(f"\n标签分布 (Top 10):")
for label, count in label_counts.most_common(10):
    print(f"  {label}: {count} ({count/len(all_true)*100:.2f}%)")

# 生成分类报告（只显示出现过的标签）
print("\nClassification Report:")
print(classification_report(all_true, all_pred, zero_division=0))

# 每个样本的详细结果
print("\n" + "="*60)
print("各样本详细结果")
print("="*60)
for r in sample_results:
    print(f"\n样本 {r['index']}:")
    print(f"  状态: {r['status']}")
    print(f"  Token数: {r['token_count']}")
    print(f"  真实标签前5: {r['true_preview']}")
    print(f"  预测标签前5: {r['pred_preview']}")

# 计算token级别的匹配统计
correct_count = sum(1 for t, p in zip(all_true, all_pred) if t == p)
print(f"\n精确匹配统计:")
print(f"  正确预测: {correct_count}/{len(all_true)} ({correct_count/len(all_true)*100:.2f}%)")

# 保存结果
eval_results = {
    "model": MODEL,
    "dataset": "ChanceFocus/flare-fnxl",
    "task": "Financial Numeric Extreme Labeling",
    "test_samples": test_size,
    "total_tokens": len(all_true),
    "accuracy": float(accuracy),
    "label_distribution": {k: v for k, v in label_counts.most_common(20)},
    "classification_report": classification_report(all_true, all_pred, zero_division=0, output_dict=True),
    "sample_results": sample_results,
    "evaluation_time": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime())
}

eval_path = os.path.join(save_dir, f"{run_tag}_10samples_results.json")
with open(eval_path, "w") as f:
    json.dump(eval_results, f, indent=2)

print(f"\n评估结果已保存 -> {eval_path}")

# 总结
print("\n" + "="*60)
print("SUMMARY - 10个测试样本")
print("="*60)
success_count = sum(1 for r in sample_results if r["status"] == "success")
print(f"API调用成功率: {success_count}/{test_size} ({success_count/test_size*100:.1f}%)")
print(f"Token-level准确率: {accuracy:.2%}")
print(f"总Token数: {len(all_true)}")

if accuracy > 0.7:
    print("\n✅ 模型表现良好，可以运行完整评估")
else:
    print("\n⚠️ 模型准确率较低，建议检查提示词或模型配置")

Step 3.5: 测试10个样本 - FLARE-FNXL
自动评估模式 | 集成推理+评估

自动评估前 10 个样本...

样本 0: 正在处理 (33 tokens)...
  ✅ 成功 | 预测前5: ['O', 'O', 'O', 'O', 'O']
样本 1: 正在处理 (35 tokens)...
  ✅ 成功 | 预测前5: ['O', 'B-ContingentConsiderationFairValue', 'I-ContingentConsiderationFairValue', 'I-ContingentConsiderationFairValue', 'O']
样本 2: 正在处理 (19 tokens)...
  ✅ 成功 | 预测前5: ['O', 'B-Year', 'O', 'O', 'O']
样本 3: 正在处理 (31 tokens)...
  ✅ 成功 | 预测前5: ['O', 'O', 'O', 'O', 'O']
样本 4: 正在处理 (34 tokens)...
  ✅ 成功 | 预测前5: ['O', 'O', 'O', 'O', 'O']
样本 5: 正在处理 (31 tokens)...
  ✅ 成功 | 预测前5: ['O', 'O', 'O', 'O', 'O']
样本 6: 正在处理 (30 tokens)...
  ✅ 成功 | 预测前5: ['O', 'O', 'O', 'O', 'O']
样本 7: 正在处理 (23 tokens)...
  ✅ 成功 | 预测前5: ['O', 'O', 'O', 'O', 'B-ReportingPeriodDate']
样本 8: 正在处理 (26 tokens)...
  ✅ 成功 | 预测前5: ['O', 'O', 'O', 'O', 'O']
样本 9: 正在处理 (21 tokens)...
  ✅ 成功 | 预测前5: ['O', 'O', 'O', 'B-FiscalYear', 'O']

EVALUATION RESULTS - o3-mini on FLARE-FNXL (10个测试样本)

总Token数: 283
唯一标签数: 33

Token-level Accuracy: 0.8198

标签分布 (Top 10):
  O: 